# 第12讲｜小学建设改变教育了吗？

一份真实数据的项目教程。目标：从1995年SUPAS公开处理版，构造地区强度×年龄队列的四格比较，再检查其解释边界。

跟做顺序：P1资料核查 → P2四格表 → P3加权DID → P4队列图 → P5设计核查 → P6备忘录。
这不是原论文全部结果的复制。图中横轴是1974年年龄，不是政策后的日历年。

## 准备
先按README建立Python环境。第一次运行需联网下载固定版本并核对SHA-256，之后可断网运行。
每个单元按顺序执行。代码完整提供；请把时间用在核对、解释与小幅修改上。

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from inpres_project import (load_data, prepare, main_sample, four_cells, did,
                            fit_wls, estimate, ar_confidence, SHA256)
OUT = Path("results")
OUT.mkdir(exist_ok=True)
plt.rcParams.update({"figure.figsize": (9, 4.8), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
TEAL, AMBER = "#197f7a", "#b87921"
def save_json(name, value):
    (OUT / name).write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                      allow_nan=False) + "\n", encoding="utf-8")


## P1｜载入与审核（课件第7页）
一行是一位1995年受访者。先查看关键列与缺失；不使用整表dropna把教育样本提前限制为工资样本。
数据源与处理边界见`DATA.md`。如果校验失败，先核查来源，不替换数据追数值。

In [ ]:
raw = load_data()
df = prepare(raw)
s = main_sample(df)
w = main_sample(df, wage=True)
audit = {"sha256": SHA256, "raw_rows": len(raw), "valid_rows": len(df),
         "education_sample": len(s), "wage_sample": len(w),
         "wage_missing_in_eligible": len(s)-len(w),
         "districts": int(s.district.nunique())}
assert (len(raw), len(df), len(s), len(w)) == (152990, 152989, 78470, 31061)
assert df.source_row.is_unique
save_json("audit.json", audit)
display(pd.Series(audit, name="audit"))
display(raw[["yeduc", "lhwage", "p504thn", "weight"]].isna().sum().to_frame("missing"))

## P2｜四格表（课件第14页）
`high`沿用来源中的`recp`；`young`为1974年2—6岁，年长比较组为12—17岁。
实际人数不加权；教育均值使用调查观测权重。它不是倾向得分权重。
先核对四格、人数与均值，再看差分。

In [ ]:
cells = four_cells(s)
cell_table = pd.DataFrame(cells)
assert len(cell_table) == 4 and int(cell_table.n.sum()) == len(s)
cell_table.to_csv(OUT / "did_cells.csv", index=False)
display(cell_table.round({"mean": 6}))
delta = did(cells)
print(f"加权四格DID = {delta:.6f} 教育年")

### 观察两段队列差额
线段连接不同人的队列均值，不是同一人的成长轨迹。纵轴使用聚焦尺度，必须在图注中说明。
保存这张图，并用一句话说明第二次相减扣除了什么。

In [ ]:
fig, ax = plt.subplots()
for high, color, label, marker in [(1, AMBER, "High intensity", "o"),
                                  (0, TEAL, "Low intensity", "s")]:
    g = cell_table[cell_table.high.eq(high)].sort_values("young")
    ax.plot(g.young, g["mean"], marker=marker, color=color, label=label)
ax.set(xticks=[0, 1], xticklabels=["Age 12-17 in 1974", "Age 2-6 in 1974"],
       ylabel="Weighted years of education", ylim=(6, 10),
       title="1995 SUPAS: cohort means (focused y-axis)")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "did_means.png", dpi=160)
plt.show()

## P3｜回归与手算核对（课件第22页）
同样本、同权重、同控制，`z`系数应等于四格DID。
按出生地区计算CR1聚类方差，并用t(G−1)构造95%区间；这没有重建完整复杂抽样设计。

In [ ]:
fit = fit_wls("yeduc ~ high + young + z", s)
result = estimate(fit, "z", s)
assert abs(result["beta"] - delta) < 1e-8
assert result["n"] == 78470 and result["clusters"] == 290
save_json("did_results.json", result)
display(pd.DataFrame([result]))

### 读出结果，而不改写结果
点估计约0.036年，95%地区聚类区间跨0。不能写成精确确认教育增加，也不能反向说已证明建校没有作用。
因果解释另外依赖跨队列平行变化、合理暴露与样本构成等条件。

## P4｜队列暴露剖面（课件第29页）
每个年龄与high交互；12岁为规范化基期。点和逐点95%区间，不是联合置信带。

In [ ]:
profile = fit_wls("yeduc ~ high * C(age74, Treatment(reference=12))", df)
cohorts = []
for age in range(24, 1, -1):
    if age == 12:
        row = {"beta": 0.0, "ci": [0.0, 0.0]}
    else:
        term = f"high:C(age74, Treatment(reference=12))[T.{float(age)}]"
        row = estimate(profile, term, df)
    cohorts.append({"age74": age, **row})
cp = pd.DataFrame(cohorts)
cp.to_csv(OUT / "cohort_profile.csv", index=False)
fig, ax = plt.subplots()
estimated = cp[cp.age74.ne(12)]
yerr = np.array([[r.beta-r.ci[0], r.ci[1]-r.beta]
                 for r in estimated.itertuples()]).T
ax.errorbar(estimated.age74, estimated.beta, yerr=yerr, fmt="o",
            color=TEAL, capsize=3, markersize=4)
ax.plot([12], [0], "D", color=AMBER, label="Reference age 12 (normalized)")
ax.axhline(0, color="gray", linewidth=1)
ax.set(xlim=(24.7, 1.3), xlabel="Age in 1974 (older to younger)",
       ylabel="High-low education gap relative to age 12 (years)",
       title="Cohort exposure profile: pointwise 95% district-cluster intervals")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "cohort_profile.png", dpi=160)
plt.show()

### 安慰剂与权重敏感性
较老队列的12—17岁与18—24岁比较，不能证明年轻队列反事实成立。年龄跨度不同，不直接作偏差扣除。
加权/不加权结果都保留，不选更显著的一行。

In [ ]:
older = df[df.age74.between(12, 24)].copy()
older["old12to17"] = older.age74.between(12, 17).astype(int)
older["placebo_z"] = older.old12to17 * older.high
placebo = estimate(fit_wls("yeduc ~ high + old12to17 + placebo_z", older),
                    "placebo_z", older)
unweighted = estimate(fit_wls("yeduc ~ high + young + z", s, False), "z", s)
save_json("did_diagnostics.json", {"placebo": placebo, "unweighted": unweighted})
display(pd.DataFrame([{"model": "weighted", **result},
                      {"model": "unweighted", **unweighted},
                      {"model": "older-cohort placebo", **placebo}]))

## P5｜设计核查（课件第36页）
打开`vibe-coding-prompts.md`的第12讲提示，核查结果列、分组、权重、聚类、样本与坐标。
无AI也可以逐项完成。把你自己的发现写入`results/design_check.md`。

不要让AI把年龄队列改成日历年份，不要复制行制造面板，不要写“未显著所以平行趋势已被证明”。

## P6｜自己的研究备忘录（课件第43页）
在`results/memo_did.md`中写150—250字：对象与比较、关键估计与区间、队列图含义、一项具体风险、一项资料需求。

## 完成检查与下讲接口
重启内核并从头运行；保留audit、四格表、结果JSON、两张图与自己的文字。
下一讲用同一资料的工资完整样本。教育主样本的DID不能直接借作工资IV的第一阶段。

来源：[Duflo (2001)](https://www.aeaweb.org/articles?id=10.1257/aer.91.4.795)；
[公开处理版说明与后续复核](https://github.com/droodman/Duflo-2001)。
本Notebook没有估计日历事件研究：缺少政策实施前后逐期结果。